# Fiche 8 — Lire un Dockerfile et un workflow GitHub Actions

| | |
|---|---|
| 🎯 **Objectif** | Expliquer ligne à ligne un Dockerfile et un workflow CI, et repérer les erreurs classiques (secret en clair, étapes dans le désordre, version non figée) |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiche 7 ; notion de terminal |
| 🔗 **Dans le projet** | Étapes 3 et 5 (Dockerfile, pipeline CI/CD) |

> **L'idée clé.** Docker emballe l'application **et son environnement** dans une **image** (le modèle figé) ; un **conteneur** est une image en train de tourner, qui se comporte pareil partout. Un workflow CI décrit, dans un fichier YAML, ce qui doit se passer à chaque modification : tester, construire, publier, dans cet ordre, et s'arrêter dès qu'une étape échoue.

*Cette fiche se lit sans Docker ni compte GitHub : on analyse les fichiers avec Python.*

## 1. Un Dockerfile minimal

In [1]:
dockerfile = '''FROM python:3.12-slim
WORKDIR /srv
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY main.py model.joblib ./
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]'''
role = {
    "FROM": "image de départ (un Linux avec Python 3.12)",
    "WORKDIR": "dossier de travail dans l'image",
    "COPY": "copie des fichiers du projet dans l'image",
    "RUN": "commande exécutée pendant la construction",
    "EXPOSE": "port sur lequel l'application écoute (documentation)",
    "CMD": "commande lancée au démarrage du conteneur",
}
for ligne in dockerfile.splitlines():
    print(f"{ligne:<75} ← {role[ligne.split()[0]]}")

FROM python:3.12-slim                                                       ← image de départ (un Linux avec Python 3.12)
WORKDIR /srv                                                                ← dossier de travail dans l'image
COPY requirements.txt .                                                     ← copie des fichiers du projet dans l'image
RUN pip install -r requirements.txt                                         ← commande exécutée pendant la construction
COPY main.py model.joblib ./                                                ← copie des fichiers du projet dans l'image
EXPOSE 8000                                                                 ← port sur lequel l'application écoute (documentation)
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]          ← commande lancée au démarrage du conteneur


Pourquoi copier `requirements.txt` et installer **avant** de copier le code ? Docker garde chaque étape en cache :
tant que les dépendances ne changent pas, une modification du code ne relance pas l'installation (longue).

## 2. Un workflow GitHub Actions

In [2]:
import yaml

workflow = yaml.safe_load('''
name: ci
on:
  push:
    branches: [main]
jobs:
  tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install -r requirements.txt pytest
      - run: pytest
  image:
    needs: tests
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: docker build -t mon-api .
''')
print(list(workflow))  # [True, 'jobs'] : PyYAML lit la clé « on » comme le booléen True !
for nom, job in workflow["jobs"].items():
    print(f"job « {nom} » (attend : {job.get('needs', 'rien')}) :", [s.get("run", s.get("uses")) for s in job["steps"]])

['name', True, 'jobs']
job « tests » (attend : rien) : ['actions/checkout@v4', 'pip install -r requirements.txt pytest', 'pytest']
job « image » (attend : tests) : ['actions/checkout@v4', 'docker build -t mon-api .']


`on` dit **quand** (à chaque push sur `main`), `jobs` dit **quoi**. `needs: tests` : le job `image` ne démarre que
si `tests` a réussi. Une commande qui échoue (code de sortie non nul, comme un `pytest` rouge) rend son job rouge et
arrête la suite.

*Curiosité* : pour PyYAML, `on` veut dire « vrai » (d'où `True` ci-dessus). GitHub, lui, le lit bien comme une clé.
Pour l'analyser en Python : `workflow[True]`.

## ⚠️ Piège : trois erreurs fréquentes, détectables automatiquement

In [3]:
import re

mauvais = yaml.safe_load('''
jobs:
  publier:
    runs-on: ubuntu-latest
    steps:
      - run: docker login -u moi -p MonMotDePasse123
      - run: docker push moi/mon-api:latest
  tests:
    runs-on: ubuntu-latest
    steps:
      - run: pytest
''')


def verifier(wf):
    alertes = []
    for nom, job in wf["jobs"].items():
        commandes = [s.get("run", "") for s in job["steps"]]
        if any(re.search(r"-p \S+|password=", c) for c in commandes):
            alertes.append(f"{nom} : mot de passe en clair dans le fichier → utiliser un secret ${{{{ secrets.X }}}}")
        tags = [t for c in commandes for t in re.findall(r"docker push \S+:(\S+)", c)]
        if tags and "needs" not in job:
            alertes.append(f"{nom} : publie sans attendre les tests → ajouter needs: tests")
        if tags and set(tags) == {"latest"}:
            alertes.append(f"{nom} : seulement le tag latest → ajouter un tag de version (ex. le commit)")
    return alertes


for a in verifier(mauvais):
    print("⚠️", a)

⚠️ publier : mot de passe en clair dans le fichier → utiliser un secret ${{ secrets.X }}
⚠️ publier : publie sans attendre les tests → ajouter needs: tests
⚠️ publier : seulement le tag latest → ajouter un tag de version (ex. le commit)


1. Un mot de passe écrit dans un fichier versionné est **public** (et reste dans l'historique git même si on
   l'efface) : on utilise les *secrets* du dépôt, ou le `GITHUB_TOKEN` fourni automatiquement.
2. Publier sans `needs` : l'image part même si les tests échouent.
3. Seulement `latest` : impossible de revenir à la version d'hier. On ajoute un tag par version (le commit).

## 3. Être prévenu d'un échec
Un job peut ne s'exécuter **que si quelque chose a échoué** avant lui : c'est l'endroit pour prévenir l'équipe.

In [4]:
alerte = yaml.safe_load('''
alerte:
  needs: [tests, image]
  if: failure()
  runs-on: ubuntu-latest
  steps:
    - run: echo "## Pipeline en échec" >> "$GITHUB_STEP_SUMMARY"
''')
print("ce job tourne seulement si :", alerte["alerte"]["if"], "| après :", alerte["alerte"]["needs"])

ce job tourne seulement si : failure() | après : ['tests', 'image']


`if: failure()` : le job ne tourne que si un job dont il dépend a échoué. `$GITHUB_STEP_SUMMARY` écrit un résumé
lisible dans l'onglet Actions. GitHub envoie aussi un e-mail à l'auteur du commit ; pour une équipe, on ajouterait
un message Slack ou Teams dans ce job.

### ✍️ À toi de jouer

Corrige le workflow `mauvais` (sous forme de dictionnaire Python ou de texte YAML) pour que `verifier` ne renvoie plus aucune alerte.

In [5]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
corrige = yaml.safe_load('''
jobs:
  tests:
    runs-on: ubuntu-latest
    steps:
      - run: pytest
  publier:
    needs: tests
    runs-on: ubuntu-latest
    steps:
      - run: echo "${{ secrets.REGISTRY_TOKEN }}" | docker login -u moi --password-stdin
      - run: docker push moi/mon-api:${GITHUB_SHA::7} && docker push moi/mon-api:latest
''')
print(verifier(corrige))   # []
```

**Pourquoi ?** le secret est lu depuis le coffre du dépôt (jamais écrit dans le fichier), la publication attend les tests, et chaque image a un tag unique. Le workflow du projet ajoute un test d'intégration : il lance le conteneur et l'appelle vraiment avant de le publier.
</details>

## ✅ À retenir

1. Dockerfile : image de départ, dépendances, code, commande de démarrage ; dépendances avant le code (cache).
2. Workflow : `on` = quand, `jobs` = quoi, `needs` = l'ordre ; une commande en échec arrête la suite.
3. Jamais de mot de passe dans un fichier : secrets du dépôt ou `GITHUB_TOKEN`.
4. Chaque image publiée a un tag de version (en plus de `latest`).
5. On teste l'image construite (conteneur lancé et appelé), pas seulement le code ; un job `if: failure()` prévient.